In [1]:
%pip install "gymnasium[box2d]" stable-baselines3[extra] torch opencv-python pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
#!/usr/bin/env python3
"""
CarRacing-v3 + PPO baseline training script (Gymnasium + Stable-Baselines3)

Implements the project decisions up to step #12:
- Gymnasium 3 with continuous actions
- Observation preprocessing: grayscale + frame stacking (4) + no resize
- PPO with Stable-Baselines3 and CnnPolicy
- 8 parallel environments for training
- EvalCallback + CheckpointCallback + TensorBoard logging
- 200k timesteps baseline run

Usage example:
    python carracing_ppo_baseline.py --total-timesteps 200000 --seed 0

Recommended install (example):
    pip install "gymnasium[box2d]" stable-baselines3[extra] torch opencv-python pyyaml
"""

from __future__ import annotations

import argparse
import json
import os
import platform
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import cv2
import gymnasium as gym
import numpy as np
import torch
import yaml
from gymnasium import spaces
from gymnasium.wrappers import TransformObservation
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecFrameStack, VecMonitor, VecTransposeImage


# -----------------------------
# Config
# -----------------------------

@dataclass
class TrainConfig:
    env_id: str = "CarRacing-v3"
    continuous: bool = True
    seed: int = 0

    # Parallelism
    n_envs: int = 8
    n_eval_envs: int = 1
    use_subproc: bool = True

    # Baseline run (step #12)
    total_timesteps: int = 200_000

    # PPO hyperparameters (step #9)
    learning_rate: float = 1e-4
    n_steps: int = 512
    batch_size: int = 128
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    ent_coef: float = 0.01
    vf_coef: float = 0.5
    max_grad_norm: float = 0.5
    use_sde: bool = False
    sde_sample_freq: int = 4
    device: str = "auto"

    # Policy kwargs
    ortho_init: bool = False
    log_std_init: float = -2.0
    pi_hidden_size: int = 256
    vf_hidden_size: int = 256

    # Observation pipeline (step #5)
    grayscale: bool = True
    n_stack: int = 4
    normalize_images_manually: bool = False

    # Logging / evaluation (steps #10/#11)
    eval_freq_steps: int = 25_000
    n_eval_episodes: int = 5
    checkpoint_freq_steps: int = 50_000
    log_root: str = "experiments/carracing_ppo"
    run_name: str | None = None

    # Optional videos
    record_video: bool = False
    video_length: int = 1_000


# -----------------------------
# Utility functions
# -----------------------------

def set_global_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def timestamp() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def make_run_dirs(cfg: TrainConfig) -> dict[str, Path]:
    run_name = cfg.run_name or f"run_{timestamp()}_seed{cfg.seed}"
    base = Path(cfg.log_root) / run_name
    paths = {
        "base": base,
        "tb": base / "tb",
        "checkpoints": base / "checkpoints",
        "best_model": base / "best_model",
        "eval": base / "eval",
        "videos": base / "videos",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


# -----------------------------
# Observation wrappers
# -----------------------------

class GrayscaleObservation(gym.ObservationWrapper):
    """Convert RGB uint8 observation (H, W, 3) to grayscale uint8 (H, W, 1)."""

    def __init__(self, env: gym.Env):
        super().__init__(env)
        assert isinstance(env.observation_space, spaces.Box)
        obs_space = env.observation_space
        assert len(obs_space.shape) == 3 and obs_space.shape[-1] == 3, (
            f"Expected channel-last RGB image, got shape={obs_space.shape}"
        )
        h, w, _ = obs_space.shape
        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=(h, w, 1),
            dtype=np.uint8,
        )

    def observation(self, obs: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        return gray[..., None].astype(np.uint8)


class Float32NormalizeObservation(gym.ObservationWrapper):
    """Convert uint8 image in [0, 255] to float32 image in [0, 1]."""

    def __init__(self, env: gym.Env):
        super().__init__(env)
        assert isinstance(env.observation_space, spaces.Box)
        old = env.observation_space
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=old.shape,
            dtype=np.float32,
        )

    def observation(self, obs: np.ndarray) -> np.ndarray:
        return (obs.astype(np.float32) / 255.0).astype(np.float32)


# -----------------------------
# Env factory
# -----------------------------

def build_single_env(cfg: TrainConfig, rank: int, run_dir: Path, training: bool) -> gym.Env:
    env = gym.make(
        cfg.env_id,
        continuous=cfg.continuous,
        render_mode="rgb_array" if cfg.record_video and not training else None,
    )

    # Seed action space and reset deterministically per env instance
    env.action_space.seed(cfg.seed + rank)
    env.reset(seed=cfg.seed + rank)

    # Observation preprocessing pipeline:
    # CarRacing-v3 -> grayscale uint8 [0,255]
    if cfg.grayscale:
        env = GrayscaleObservation(env)

    # Episode statistics for individual envs
    monitor_file = run_dir / f"monitor_{'train' if training else 'eval'}_{rank}.csv"
    env = Monitor(env, filename=str(monitor_file))
    return env


def make_env_fn(cfg: TrainConfig, rank: int, run_dir: Path, training: bool):
    def _init() -> gym.Env:
        return build_single_env(cfg, rank, run_dir, training)
    return _init


# -----------------------------
# Vec env builders
# -----------------------------

def make_train_vec_env(cfg: TrainConfig, paths: dict[str, Path]):
    env_fns = [make_env_fn(cfg, i, paths["base"], training=True) for i in range(cfg.n_envs)]
    if cfg.use_subproc and cfg.n_envs > 1:
        vec_env = SubprocVecEnv(env_fns, start_method="spawn")
    else:
        vec_env = DummyVecEnv(env_fns)

    vec_env = VecMonitor(vec_env, filename=str(paths["base"] / "vec_monitor_train.csv"))
    vec_env = VecFrameStack(vec_env, n_stack=cfg.n_stack)
    vec_env = VecTransposeImage(vec_env)
    return vec_env


def make_eval_vec_env(cfg: TrainConfig, paths: dict[str, Path]):
    env_fns = [make_env_fn(cfg, 10_000 + i, paths["base"], training=False) for i in range(cfg.n_eval_envs)]
    vec_env = DummyVecEnv(env_fns)
    vec_env = VecMonitor(vec_env, filename=str(paths["base"] / "vec_monitor_eval.csv"))
    vec_env = VecFrameStack(vec_env, n_stack=cfg.n_stack)
    vec_env = VecTransposeImage(vec_env)
    return vec_env


# -----------------------------
# Model builder
# -----------------------------

def build_model(cfg: TrainConfig, train_env, paths: dict[str, Path]) -> PPO:
    policy_kwargs: dict[str, Any] = {
        "ortho_init": cfg.ortho_init,
        "log_std_init": cfg.log_std_init,
        "net_arch": {"pi": [cfg.pi_hidden_size], "vf": [cfg.vf_hidden_size]},
    }

    model = PPO(
        policy="CnnPolicy",
        env=train_env,
        learning_rate=cfg.learning_rate,
        n_steps=cfg.n_steps,
        batch_size=cfg.batch_size,
        n_epochs=cfg.n_epochs,
        gamma=cfg.gamma,
        gae_lambda=cfg.gae_lambda,
        clip_range=cfg.clip_range,
        ent_coef=cfg.ent_coef,
        vf_coef=cfg.vf_coef,
        max_grad_norm=cfg.max_grad_norm,
        use_sde=cfg.use_sde,
        sde_sample_freq=cfg.sde_sample_freq,
        policy_kwargs=policy_kwargs,
        tensorboard_log=str(paths["tb"]),
        verbose=1,
        seed=cfg.seed,
        device=cfg.device,
    )
    return model


# -----------------------------
# Callbacks
# -----------------------------

def build_callbacks(cfg: TrainConfig, eval_env, paths: dict[str, Path]) -> CallbackList:
    # SB3 callback frequencies operate in calls to env.step(); with n_envs parallel envs,
    # convert desired real timesteps to callback frequency in vectorized steps.
    eval_freq = max(cfg.eval_freq_steps // cfg.n_envs, 1)
    checkpoint_freq = max(cfg.checkpoint_freq_steps // cfg.n_envs, 1)

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(paths["best_model"]),
        log_path=str(paths["eval"]),
        eval_freq=eval_freq,
        n_eval_episodes=cfg.n_eval_episodes,
        deterministic=True,
        render=False,
        verbose=1,
    )

    checkpoint_callback = CheckpointCallback(
        save_freq=checkpoint_freq,
        save_path=str(paths["checkpoints"]),
        name_prefix="ppo_carracing",
        save_replay_buffer=False,
        save_vecnormalize=False,
        verbose=1,
    )

    return CallbackList([checkpoint_callback, eval_callback])


# -----------------------------
# Config export
# -----------------------------


def to_yaml_safe(obj):
    if isinstance(obj, dict):
        return {str(k): to_yaml_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_yaml_safe(x) for x in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            return str(obj)
    return str(obj)

def export_config(cfg: TrainConfig, paths: dict[str, Path]) -> None:
    payload = asdict(cfg)
    payload["timestamp"] = str(datetime.now().isoformat())
    payload["python_version"] = str(platform.python_version())
    payload["platform"] = str(platform.platform())
    payload["torch_version"] = str(torch.__version__)
    payload["cuda_available"] = bool(torch.cuda.is_available())

    # Optional imports guarded for portability
    try:
        import stable_baselines3
        payload["stable_baselines3_version"] = str(stable_baselines3.__version__)
    except Exception:
        payload["stable_baselines3_version"] = "unknown"

    try:
        import gymnasium
        payload["gymnasium_version"] = str(gymnasium.__version__)
    except Exception:
        payload["gymnasium_version"] = "unknown"

    payload = to_yaml_safe(payload)

    with open(paths["base"] / "config.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(payload, f, sort_keys=False, allow_unicode=True)

    with open(paths["base"] / "config.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)




def get_config() -> TrainConfig:

    seed = 0 # default 0
    total_timesteps = 200000 # default 200000
    n_envs = 8 # default 8
    run_name = None # default None
    log_root = "experiments/carracing_ppo" # default "experiments/carracing_ppo"
    device = "auto" # default "auto"
    dummy_vec = True # default False

    return TrainConfig(
        seed=seed,
        total_timesteps=total_timesteps,
        n_envs=n_envs,
        run_name=run_name,
        log_root=log_root,
        device=device,
        use_subproc=not dummy_vec
    )


# -----------------------------
# Main
# -----------------------------

def main() -> None:
    cfg = get_config()
    set_global_seeds(cfg.seed)
    paths = make_run_dirs(cfg)
    export_config(cfg, paths)

    print("=" * 80)
    print("CarRacing-v3 PPO baseline")
    print(f"Run dir:          {paths['base']}")
    print(f"Seed:             {cfg.seed}")
    print(f"Total timesteps:  {cfg.total_timesteps}")
    print(f"Parallel envs:    {cfg.n_envs}")
    print(f"Eval every:       {cfg.eval_freq_steps} env steps")
    print(f"Checkpoint every: {cfg.checkpoint_freq_steps} env steps")
    print("=" * 80)

    train_env = make_train_vec_env(cfg, paths)
    eval_env = make_eval_vec_env(cfg, paths)

    model = build_model(cfg, train_env, paths)
    callbacks = build_callbacks(cfg, eval_env, paths)

    try:
        model.learn(
            total_timesteps=cfg.total_timesteps,
            callback=callbacks,
            progress_bar=True,
            tb_log_name="ppo_carracing",
            reset_num_timesteps=True,
        )

        final_model_path = paths["base"] / "final_model.zip"
        model.save(str(final_model_path))
        print(f"Saved final model to: {final_model_path}")
        print(f"Best model path:      {paths['best_model']}")
        print(f"TensorBoard log dir:  {paths['tb']}")

    finally:
        train_env.close()
        eval_env.close()

In [3]:
main()

CarRacing-v3 PPO baseline
Run dir:          experiments\carracing_ppo\run_20260422_193928_seed0
Seed:             0
Total timesteps:  200000
Parallel envs:    8
Eval every:       25000 env steps
Checkpoint every: 50000 env steps


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
Logging to experiments\carracing_ppo\run_20260422_193928_seed0\tb\ppo_carracing_1


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

-----------------------------
| time/              |      |
|    fps             | 66   |
|    iterations      | 1    |
|    time_elapsed    | 61   |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -45.3       |
| time/                   |             |
|    fps                  | 54          |
|    iterations           | 2           |
|    time_elapsed         | 151         |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.011781095 |
|    clip_fraction        | 0.171       |
|    clip_range           | 0.2         |
|    entropy_loss         | 1.75        |
|    explained_variance   | -0.000295   |
|    learning_rate        | 0.0001      |
|    loss                 | 0.535       |
|    n_updates            | 10          |
|    policy_gradient_loss | 0.0005

...

Episode length: 753.40 +/- 206.74

----------------------------------------
| eval/                   |            |
|    mean_ep_length       | 753        |
|    mean_reward          | 307        |
| time/                   |            |
|    total_timesteps      | 200000     |
| train/                  |            |
|    approx_kl            | 0.11596534 |
|    clip_fraction        | 0.216      |
|    clip_range           | 0.2        |
|    entropy_loss         | 1.8        |
|    explained_variance   | 0.962      |
|    learning_rate        | 0.0001     |
|    loss                 | 2.75       |
|    n_updates            | 480        |
|    policy_gradient_loss | -0.00433   |
|    std                  | 0.133      |
|    value_loss           | 11.9       |
----------------------------------------


New best mean reward!

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 979      |
|    ep_rew_mean     | 226      |
| time/              |          |
|    fps             | 48       |
|    iterations      | 49       |
|    time_elapsed    | 4174     |
|    total_timesteps | 200704   |
---------------------------------


Saved final model to: experiments\carracing_ppo\run_20260422_193928_seed0\final_model.zip
Best model path:      experiments\carracing_ppo\run_20260422_193928_seed0\best_model
TensorBoard log dir:  experiments\carracing_ppo\run_20260422_193928_seed0\tb
